In [2]:
import sys
if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

import xml.etree.ElementTree as ET
import requests
import csv
import os
import re
import json
import base64
import hashlib
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from bs4 import BeautifulSoup
from tqdm import tqdm

try:
    import pymupdf as fitz  # PyMuPDF
except ImportError:
    try:
        import fitz
    except ImportError:
        print("Please install PyMuPDF: pip install PyMuPDF")

try:
    from pdf2image import convert_from_bytes
    import pytesseract
    HAS_OCR = True
except Exception:
    HAS_OCR = False

# Constants
OAI_BASE = "https://digital.library.unt.edu/oai/"
COLLECTION_SET = "collection:IIPCM"
NAMESPACES = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "untl": "http://digital2.library.unt.edu/untl/",
}
OUTPUT_FILE = "iipcm_extracted_content.csv"
MAX_WORKERS = 8

# Single session to preserve authenticated cookies
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
})

altcha_lock = threading.Lock()
csv_lock = threading.Lock()

def safe_get(url, stream=False):
    """Wrapper for requests.get that automatically solves Altcha proof-of-work challenges if triggered."""
    try:
        r = session.get(url, timeout=25, stream=stream)
    except Exception:
        return None

    # Check for Altcha bot challenge
    if not stream and ("altcha" in r.text.lower() or "validating your request" in r.text.lower()):
        with altcha_lock:
            # Re-verify inside lock in case another thread solved it in the meantime
            try:
                r = session.get(url, timeout=25)
            except Exception:
                return None
            if "altcha" not in r.text.lower() and "validating your request" not in r.text.lower():
                return r

            print(f"\n[ALTCHA] Challenge triggered for: {url}. Solving...")
            try:
                csrf_match = re.search(r'name="csrfmiddlewaretoken"\s+value="([^"]+)"', r.text)
                if not csrf_match:
                    print("[ALTCHA] Could not find CSRF token on Altcha page.")
                    return r
                csrf_token = csrf_match.group(1)

                algo_match = re.search(r'algorithm:\s*"([^"]+)"', r.text)
                challenge_match = re.search(r'challenge:\s*"([^"]+)"', r.text)
                salt_match = re.search(r'salt:\s*"([^"]+)"', r.text)
                sig_match = re.search(r'signature:\s*"([^"]+)"', r.text)
                max_match = re.search(r'maxnumber:\s*"([^"]+)"', r.text)

                if not all([algo_match, challenge_match, salt_match, sig_match, max_match]):
                    print("[ALTCHA] Could not extract challenge details.")
                    return r

                algorithm = algo_match.group(1)
                challenge_hash = challenge_match.group(1)
                salt = salt_match.group(1)
                signature = sig_match.group(1)
                maxnumber = int(max_match.group(1))

                solved_num = None
                for n in range(maxnumber + 1):
                    h = hashlib.sha256((salt + str(n)).encode()).hexdigest()
                    if h == challenge_hash:
                        solved_num = n
                        break

                if solved_num is None:
                    print("[ALTCHA] Failed to solve puzzle.")
                    return r

                altcha_payload = {
                    "algorithm": algorithm,
                    "challenge": challenge_hash,
                    "number": solved_num,
                    "salt": salt,
                    "signature": signature
                }
                altcha_payload_str = json.dumps(altcha_payload, separators=(',', ':'))
                altcha_b64 = base64.b64encode(altcha_payload_str.encode()).decode()

                submit_url = "https://digital.library.unt.edu/dam/submit/"
                headers = {
                    "X-CSRFToken": csrf_token,
                    "Origin": "https://digital.library.unt.edu",
                    "Referer": url
                }
                payload_data = {
                    "csrfmiddlewaretoken": csrf_token,
                    "altcha": altcha_b64
                }

                submit_resp = session.post(submit_url, data=payload_data, headers=headers, timeout=20)
                if submit_resp.status_code == 200 and "success" in submit_resp.text:
                    print("[ALTCHA] Solved and session authenticated successfully!")
                    r = session.get(url, timeout=25, stream=stream)
                else:
                    print(f"[ALTCHA] Failed to submit: {submit_resp.text}")
            except Exception as e:
                print(f"[ALTCHA ERROR] {e}")

    return r

def extract_pdf_text(pdf_url):
    try:
        response = safe_get(pdf_url)
        if not response or response.status_code != 200:
            return ""
        content_type = response.headers.get("Content-Type", "")
        if "application/pdf" not in content_type and not response.content.startswith(b"%PDF"):
            return ""
        pdf_bytes = response.content

        # 1. Fast PyMuPDF extraction
        try:
            doc = fitz.open(stream=pdf_bytes, filetype="pdf")
            text = "\n".join([page.get_text() for page in doc])
            if text.strip():
                return text.strip()
        except Exception:
            pass

        # 2. Fast OCR fallback (capped to first 3 pages at 100 DPI)
        if HAS_OCR:
            try:
                images = convert_from_bytes(pdf_bytes, dpi=100, first_page=1, last_page=3)
                text = ""
                for img in images:
                    text += pytesseract.image_to_string(img) + "\n"
                return text.strip()
            except Exception:
                pass

        return ""
    except Exception:
        return ""

def extract_vtt_transcript(vtt_url):
    try:
        response = safe_get(vtt_url)
        if not response or response.status_code != 200:
            return ""
        lines = response.text.splitlines()
        transcript = []
        for line in lines:
            line = line.strip()
            if (
                line.startswith("WEBVTT") or
                line.startswith("NOTE") or
                "-->" in line or
                line.isdigit() or
                (":" in line and line.lower().startswith("vtt_")) or
                line == ""
            ):
                continue
            transcript.append(line)
        return " ".join(transcript)
    except Exception:
        return ""

def harvest_oai_records():
    print("[HARVEST] Harvesting metadata using OAI-PMH (untl)...")
    records = []
    token = None

    while True:
        params = {
            "verb": "ListRecords",
            "metadataPrefix": "untl",
            "set": COLLECTION_SET
        } if not token else {
            "verb": "ListRecords",
            "resumptionToken": token
        }

        resp = safe_get(OAI_BASE + "?" + "&".join([f"{k}={v}" for k, v in params.items()]))
        if not resp:
            break
        root = ET.fromstring(resp.content)

        for r in root.findall(".//oai:record", NAMESPACES):
            header = r.find("oai:header", NAMESPACES)
            if header is None or header.attrib.get("status") == "deleted":
                continue

            meta = r.find(".//untl:metadata", NAMESPACES)
            if meta is None:
                continue

            item_url_el = meta.find('.//untl:identifier[@qualifier="itemURL"]', NAMESPACES)
            ark_url = item_url_el.text.strip() if item_url_el is not None and item_url_el.text else ""
            if not ark_url:
                continue

            title_el = meta.find("untl:title", NAMESPACES)
            title = title_el.text.strip() if title_el is not None and title_el.text else ""

            date_el = meta.find("untl:date", NAMESPACES)
            date = date_el.text.strip() if date_el is not None and date_el.text else ""

            creators = []
            for creator_el in meta.findall("untl:creator", NAMESPACES):
                name_el = creator_el.find("untl:name", NAMESPACES)
                info_el = creator_el.find("untl:info", NAMESPACES)
                name = name_el.text.strip() if name_el is not None and name_el.text else ""
                info = info_el.text.strip() if info_el is not None and info_el.text else ""
                if name:
                    if info:
                        creators.append(f"{name} ({info})")
                    else:
                        creators.append(name)
            creator_str = "; ".join(creators)

            subjects = [el.text.strip() for el in meta.findall("untl:subject", NAMESPACES) if el.text]
            subject_str = "; ".join(subjects)

            desc_el = meta.find("untl:description", NAMESPACES)
            description = desc_el.text.strip() if desc_el is not None and desc_el.text else ""

            conf_el = meta.find('.//untl:source[@qualifier="conference"]', NAMESPACES)
            conference = conf_el.text.strip() if conf_el is not None and conf_el.text else ""
            if conference:
                description = f"{description}\nConference: {conference}"

            type_el = meta.find("untl:resourceType", NAMESPACES)
            item_type = type_el.text.strip() if type_el is not None and type_el.text else "text"

            record = {
                "ark_url": ark_url,
                "title": title,
                "date": date,
                "creator": creator_str,
                "subject": subject_str,
                "description": description,
                "item_type": item_type,
            }
            records.append(record)

        token_el = root.find(".//oai:resumptionToken", NAMESPACES)
        token = token_el.text.strip() if token_el is not None and token_el.text else None
        if not token:
            break

    print(f"[HARVEST] Retrieved {len(records)} metadata records.")
    return records

def resolve_pdf_link(folder_url):
    try:
        res = safe_get(folder_url)
        if not res:
            return ""
        soup = BeautifulSoup(res.text, "html.parser")
        for link in soup.find_all("a"):
            href = link.get("href", "")
            if href.endswith(".pdf"):
                return requests.compat.urljoin(folder_url, href)
    except Exception:
        pass
    return ""

def process_record(record):
    ark_url = record["ark_url"]
    item_type = record["item_type"].lower()
    text = ""
    file_url = ""

    try:
        res = safe_get(ark_url)
        if res and res.status_code == 200:
            soup = BeautifulSoup(res.text, "html.parser")
            links = soup.find_all("a")

            for link in links:
                href = link.get("href", "")
                full_url = requests.compat.urljoin(ark_url, href)

                if "video" in item_type and href.endswith(".vtt"):
                    text = extract_vtt_transcript(full_url)
                    file_url = full_url
                    break
                elif "video" not in item_type and (href.endswith(".pdf") or "high_res_d/" in href or "/m2/" in href):
                    if href.endswith("/"):
                        pdf_resolved = resolve_pdf_link(full_url)
                        if pdf_resolved:
                            text = extract_pdf_text(pdf_resolved)
                            file_url = pdf_resolved
                            break
                    else:
                        text = extract_pdf_text(full_url)
                        file_url = full_url
                        break
    except Exception:
        pass

    record["source_url"] = file_url
    record["full_text"] = text.replace("\r", "").replace("\n", "\\n")
    return record

def main():
    keys = ["ark_url", "title", "date", "creator", "subject", "description", "item_type", "source_url", "full_text"]

    # 1. Harvest metadata
    records = harvest_oai_records()

    # 2. Check for existing progress (only skip if full_text is non-empty)
    existing_records = {}
    if os.path.exists(OUTPUT_FILE):
        print(f"[RESUME] Found existing {OUTPUT_FILE}. Checking completed documents...")
        try:
            with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("ark_url") and row.get("full_text") and len(row["full_text"].strip()) > 0:
                        existing_records[row["ark_url"]] = row
            print(f"[RESUME] Loaded {len(existing_records)} completed documents with text. These will be skipped.")
        except Exception as e:
            print(f"[RESUME] Could not load existing file: {e}")

    # 3. Authenticate session upfront before spawning threads
    print("[AUTH] Authenticating session with UNT server...")
    if records:
        safe_get(records[0]["ark_url"])
    print("[AUTH] Session authenticated!")

    to_process = [r for r in records if r["ark_url"] not in existing_records]
    processed = list(existing_records.values())
    print(f"[STATUS] Total: {len(records)} | Skipped: {len(processed)} | To Process: {len(to_process)}")

    if to_process:
        file_mode = "a" if os.path.exists(OUTPUT_FILE) and existing_records else "w"
        if file_mode == "w":
            with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=keys)
                writer.writeheader()

        with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as csv_out:
            writer = csv.DictWriter(csv_out, fieldnames=keys)

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = {executor.submit(process_record, rec): rec for rec in to_process}

                for future in tqdm(as_completed(futures), total=len(to_process), desc="Processing Records"):
                    try:
                        res = future.result()
                    except Exception:
                        res = futures[future]
                        res["source_url"] = ""
                        res["full_text"] = ""

                    processed.append(res)
                    with csv_lock:
                        writer.writerow(res)
                        csv_out.flush()

    print(f"\n[DONE] Successfully saved {len(processed)} records to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

[HARVEST] Harvesting metadata using OAI-PMH (untl)...
[HARVEST] Retrieved 626 metadata records.
[AUTH] Authenticating session with UNT server...

[ALTCHA] Challenge triggered for: https://digital.library.unt.edu/ark:/67531/metadc1393823/. Solving...
[ALTCHA] Solved and session authenticated successfully!
[AUTH] Session authenticated!
[STATUS] Total: 626 | Skipped: 0 | To Process: 626


Processing Records: 100%|██████████| 626/626 [39:18<00:00,  3.77s/it]  


[DONE] Successfully saved 626 records to iipcm_extracted_content.csv
